# testsimul.pro

Translation to Python from Andrei's IDL code.\
All this notebook should be a single .py file\
but each cell contains different .py files and functions

In [ ]:
import codecs
import json
import logging
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.animation as animation
import numpy as np
import os
import pandas as pd
import scipy as sci
from scipy import optimize
import sys
from tqdm.notebook import tqdm


from astropy.io import fits
import datetime
from IPython.display import display, clear_output
from scipy.signal import detrend, find_peaks
from scipy.ndimage import zoom, map_coordinates, shift as ndshift
from scipy.special import factorial
import zernike

In [ ]:
logger = logging.getLogger('testsimul')
logger.setLevel('DEBUG')

### sim1.par

In [ ]:
#param,
site = 'LaSerena'       # site
lon = -71.2425          # longitude
lat = -29.91744         # latitude
d = 0.304               # aperture diam., m
effl = 1.20845          # effective focal length, m
eps = 0.7               # central obstruction  
pixel = 0.017664		# pixel scale, arcsec
wavelen = 0.6e-6        # wavelength, m
pdist = 1200            # defocus expressed as conj. height, m
ringradpix = 20         # default ring inner radius, pixels
drad = 1.5              # mask width in lambda/(0.5*D*(1-eps)) units
mmax = 20               # max order of angular signals
nsect = 8               # number of sectors for radius calculation
interpol = 1            # sub-pixel shifts?
ron = 0                 # readout noise, el
dirdat = 'D:/RINGSS/code5/'                      # fits data cubes
datfile = 'D:/RINGSS/code5/simulateddata.sav'    # results file
weightfile = 'weight-simu.idl'                   # weights
qedata = 'qesimul.txt'                           # spectral response
zmat = 1                                         # saturation correction

In [ ]:
seeing = 1      # Seeing in arcsec at 0.5 micron
zlow = 500      # Lower layer altitude in meters
zhigh = 10500   # High layer altitude in meters
highfrac = 0.1  # (No comment yet)
starmag = 2     # Star magnitude
gain = 0        # Camera gain setting

In [ ]:
pixsize = 6.9e-6
pixscale =  (pixsize) / effl * 206265
print(f'Calculated pixel scale', pixscale)


r0 = 0.98 * wavelen / seeing * 206265.0 # Fried parameter in meters
tint = (r0 ** (-5.0 / 3.0)) / 0.423 * ((0.5 * wavelen / np.pi) ** 2)
print(f' Input r0 = {r0:2f} and J (turbulence integral) = {tint}')

In [ ]:
#par = with open data sim1.par file as data blablabla

### Simatm

In [ ]:
#args = p.parse_args()

#pixel = 0.017664 # Grid pixel in meters (pixel size? ccd size?)
wavelen = wavelen
ngrid = 1024
size = 2 * ngrid * pixel

print('Simulating atmosphere')

logger.debug(f'size: {size} \n ngrid: {ngrid}')

tint0 = (r0 ** (-5 / 3)) / 0.423 * ((0.5 * wavelen / np.pi) ** 2) # Turbulence integral in meters^1/3
see = (tint0 / 6.83e-13) ** (0.6)
tinthigh = tint0 * highfrac            # High layer integral
tintlow = tint0 * (1 - highfrac)       # Low layer integral
if zlow >= zhigh:
    raise ValueError(f"Inconsistency: zlow ({zlow}) can't be greater or equal than zhigh ({zhigh})")

r0high = (0.423 * (0.5 * wavelen / np.pi)**(-2) * tinthigh)**(-3 / 5)  # Fried param for high layer
r0low = (0.423 * (0.5 * wavelen / np.pi)**(-2) * tintlow)**(-3 / 5)    # Fried param for low layer

print(f'Grid size (meters): {size} \n Fried parameters (meters) [low, high]: [{r0low, r0high}]')
print(f'Integrals (meters^1/3) [low, high]: [{tintlow, tinthigh}] \n Altitudes (meters) {zlow, zhigh}')
print(f'Seeing (arcsec): {see}')

# For phase simulations
if highfrac != 0:
    facthigh = np.sqrt(0.023) * ((size/r0high) ** (5/6))
else:
    facthigh = 0
factlow = np.sqrt(0.023) * ((size/r0low) ** (5/6))

# Create grid and radial distance from center in pixels
y, x = np.ogrid[:ngrid * 2, :ngrid * 2]
r = np.sqrt((x - ngrid)**2 + (y - ngrid)**2, dtype=np.float64)
r[ngrid, ngrid] = 1e-3

# Create Fresnel filters
farg = np.pi * wavelen / (size ** 2) * r ** 2

# Set seed for rng, if seed0 is provided, use it. Otherwise, let the OS provide a random one
if 'seed0' in globals() and seed0 is not None:
    rng = np.random.default_rng(seed0)
    logger.debug(f'Simulation on fixed seed: {seed0}.')
else:
    rng = np.random.default_rng()
    logger.debug('Simulation on random seed.')

# Simulate turbulence in high layer
if highfrac > 0:
    print('Simulating high layer')
    rng_complex = rng.normal(size=(ngrid*2, ngrid*2)) +1j * rng.normal(size=(ngrid*2, ngrid*2))
    
    tmp_fourier = facthigh * (r ** (-11 / 6)) * rng_complex
    tmp_fourier[ngrid, ngrid] = 0 + 0j
    tmp_shifted = np.fft.ifftshift(tmp_fourier)
    tmp_spatial = np.fft.ifft2(tmp_shifted) * ((ngrid*2) ** 2)
    tmp_phase = np.fft.fftshift(tmp_spatial).real

    # Simulate phase
    u1 = np.exp(1j * tmp_phase)
    
    # Propagate using Angular spectrum method
    if (zhigh - zlow) > 0:
        print('Propagating to low layer')
        dz = zhigh - zlow
        H_transfer = np.exp(-1j * farg * dz)
        u1_fourier = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(u1)))
        tmp_prop = H_transfer * u1_fourier
        u1 = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(tmp_prop)))
else:
    u1 = np.ones((ngrid*2, ngrid*2), dtype=np.complex128)

# Simulate Low layer
print('Simulating low layer')
rng_complex_low = rng.normal(size=(ngrid*2, ngrid*2)) + 1j * rng.normal(size=(ngrid*2, ngrid*2))

tmp_fourier_low = factlow * (r ** (-11 / 6)) * rng_complex_low
tmp_fourier_low[ngrid, ngrid] = 0 + 0j

tmp_shifted_low = np.fft.ifftshift(tmp_fourier_low)
tmp_spatial_low = np.fft.ifft2(tmp_shifted_low) * ((ngrid*2) ** 2)
tmp_phase_low = np.fft.fftshift(tmp_spatial_low).real

# Apply low layer phase distortion
u1 *= np.exp(1j * tmp_phase_low)

# Propagate to ground
print('Propagating to ground')
H_ground = np.exp(-1j * farg * zlow)
u1_fourier_ground = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(u1)))
tmp_ground = H_ground * u1_fourier_ground
u1 = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(tmp_ground)))

np.savez(
    'atm.npz',
    u1=u1,
    ngrid=ngrid,
    pixel=pixel,
    wavelen=wavelen,
    see=see,
    r0=r0,
    highfrac=highfrac,
    zhigh=zhigh,
    zlow=zlow
    )

# Diagnostics
scint = np.sum((np.abs(u1)**2 - 1)**2) / (2 * ngrid) ** 2
rytov = 19.22 * (wavelen ** (-7 / 6)) * ((zlow ** (5 / 6)) * tintlow + (zhigh**(5 / 6)) * tinthigh)
intensity = np.abs(u1) ** 2

print(f'Rytov variance, simulated: {rytov}, {scint}')

plt.figure(figsize=(7, 7))
plt.imshow(intensity, cmap='GnBu', origin='lower', norm=colors.LogNorm())
plt.title('Simulated atmosphere')
plt.tight_layout()
plt.savefig('atmsim.jpg', dpi=300, format='jpg')
plt.show()

### ringsim.pro

In [ ]:
with np.load('atm.npz') as data:
    u1        = data['u1']
    ngrid     = data['ngrid']
    pixel     = data['pixel']
    wavelen   = data['wavelen']
    see       = data['see']
    r0        = data['r0']
    highfrac  = data['highfrac']
    zlow      = data['zlow']
    zhigh     = data['zhigh']

# Hard-coded Parameters, these change with args.parse
starmag = starmag if 'starmag' in locals() else 2 # star magnitude
ron = ron if 'ron' in locals() else 1             # electrons, read out noise 
d = d                                             # meters, mirror diameter
eps = eps                                         # central obscuration 
pdist = pdist                                     # meters, H (Conjugation distance)
texp = 1e-3                                       # seconds, exposure time
tacc = 1                                          # seconds, accumulation time
wind = wind if 'wind' in locals() else 10         # m/s, wind speed
oversamp = True                                   # check to oversample
blur = True
jitter = 0

# Apertures move over the screen mostly in x-direction, but slide
# in y-direction by SLIDE meters par grid length
size = 2 * ngrid * pixel                          # grid size in arcseconds?
slide = 0.205
alpha = slide / size                              # tangent of slide angle

print(f'Loaded {data}')
print('starmag, ron, d, eps, pdist, wind, size, ngrid')
print(starmag, ron, d, eps, pdist, wind, size, ngrid)


if blur:
    nblur = math.floor(wind * texp / pixel + 0.5)
        # ^math.floor^ returns integer like int(np.floor())
    print(f'Averaging atmospheric screens, N={nblur}')
    if nblur > 1:
        tmp = np.copy(u1)
        for k in range(1, nblur):
            tmp += np.roll(u1, shift=k, axis=1)  # Shift in x-direction
        u1 = tmp / nblur

if 'zlow' not in locals() or zlow is None:
    zlow = zhigh

# Calculate seeing in arcseconds (206265 converts radians to arcseconds)
seeing = 0.98 * wavelen / r0 * 206265
print(f'Seeing (arcsec): {round(seeing,5)} arcsec \n Layers at [{zlow}, {zhigh}] meters with high fraction {highfrac}')
print(f'Screen size (meters): {2 * ngrid * pixel} \n Pixel size (meters): {pixel}')

# Total turbulence integral J (m^(1/3))
tint = (r0 ** (-5 / 3)) / 0.423 * ((0.5 * wavelen / np.pi) ** 2)
print(f'Input r0 (meters): {r0}, J: {tint}')

niter = math.floor(tacc / texp)           # Steps
jstep = math.floor(wind * texp / pixel + 0.5) # Integer pixel shift
if jstep < 1:
    jstep = 1
windef = jstep * pixel / texp


print(f'Screen shift per exposure: {jstep} pixels')
print(f'Effective wind speed: {windef} m/s')
print(f'Total iterations to simulate: {niter}\n')

# Define grid size and oversampling
nap = math.floor(ngrid / 2)
print(f'Aperture grid, pixels: {nap}')

d1 = wavelen / pixscale * 206265                                        # Pupil match pixels
logger.debug(f'd1: {d1}')
d2 = wavelen / pixel * 206265                                        # Pupil match pixels
logger.debug(f'd2: {d2}')

if oversamp: 
    npixperpix = 2 ** (math.floor(math.log2(1.5 * d / d1)) + 1) # Oversample
else:
    npixperpix = 1
print(f'Pixel per pixel: {npixperpix}')

nscr = 2 ** (math.floor(math.log2(1.5 * d / pixel)) + 1)            # Screen size in pixels
print(f'Screen size in pixels: {nscr}')

ksamp = max(int(nap // nscr), 1)
print(f'Over-sampling factor: {ksamp}')


# Pixel and ring sizes
asperpix = pixel # This shouldnt be correct, pixel is in meters not arcsec
# asperpix = 206265 * wavelen / (nap * pixel / ksamp) * npixperpix
nccd = int(nap // npixperpix) # Number of CCD pixels
rradiuspix = 0.85 * d * (1 + eps) / (4 * pdist) * 206265 / asperpix

print(f'Re-sampled pixel size [m]: {pixel / ksamp:.6e}')
print(f'CCD size & pixel [arcsec]: {nccd}, {asperpix}')
print(f'Nominal ring radius [pix, arcsec]: {rradiuspix:.3f}, {rradiuspix * asperpix:.3f}')

# AAAAH this doesn't make sense
if abs(asperpix / pixel - 1.0) > 0.05:
    print(f'Discrepant pixel scale in par! {pixel}')
    print(f'To match, use SIMATM with pixel of {pixel * asperpix / pixel}')
    raise ValueError('Pixel scale discrepancy exceeds 5%. Aborting.')

# Circular image shifts
if jitter > 0:
    x1d = np.arange(nccd, dtype=np.float64)
    omega = (3.3 / niter) * (2 * np.pi)  # angular frequency

# Star
if starmag != 0:
    BW = 0.26                      # effective bandwidth
    phot_con = 1e11                # constant representing photons/sec/m^2 for a Mag 0 star at the top of the atmosphere
    starph = phot_con * texp * BW * 10**(-0.4 * starmag) * np.pi * (d/2)**2 * (1 - eps)**2 
    
print(f'Stellar photons per exposure and Star Magnitude: {round(starph,2),starmag}')
print(f'Readout noise (e-): {ron}')
print(f'Jitter = {jitter}')

# Prepare the aperture mask
apert = np.zeros((nap, nap))
y, x = np.ogrid[:nap, :nap]
r = np.hypot(x - nap / 2, y - nap / 2) # Radial distance in pixels
radpix = d * 0.5 / pixel * ksamp
inside = (r <= radpix) & (r >= radpix * eps)
apert[inside] = 1
# apert[0 : nap // 2, nap // 2 - 10 : nap // 2 + 10] = 0 # Optional sector mask test

if logger.isEnabledFor(logging.DEBUG):
    print(f'radpix: ({d} * 0.5) / {pixel}) * {ksamp} = {radpix} \n n inside: {np.sum(apert)}')
    plt.figure(figsize=(6, 6))
    plt.imshow(apert, cmap='gray', origin='lower')
    plt.title(f'Debug Aperture Mask (nap={nap}, radpix={round(radpix,2)})')
    plt.colorbar(ticks = (0,1))
    plt.xlabel('X [pixels]')
    plt.ylabel('Y [pixels]')
    plt.show()
    plt.clf

# Add defocus and spherical, a4 negative for intrafocal
a4 = (d ** 2 / (wavelen * pdist)) * (np.pi / (8 * np.sqrt(3)))
a11 = -0.1 * a4  # Matching spherical aberration
rho = r / radpix

tmp = a11 * np.sqrt(5) * (6.0 * (rho**4) - 6 * (rho**2))
tmp += a4 * 2.0 * np.sqrt(3) * ((rho**2) - 0.5)

print(f'Nominal a4, a11 [rad]: {a4:}, {a11}')

if logger.isEnabledFor(logging.DEBUG):
    plt.figure(figsize=(6, 6))
    plt.imshow(tmp, cmap='gray', origin='lower', norm=colors.LogNorm())
    plt.title(f'Debug Zernike applied (nap={nap}, radpix={round(radpix/ksamp, 2)})')
    plt.colorbar()
    plt.xlabel('X [pixels]')
    plt.ylabel('Y [pixels]')
    plt.show()
    plt.clf

# Undistorted image
fresnel = np.exp(1j * tmp) * apert

# Mirror the correct shift order for consistency
pupil_shifted = np.fft.ifftshift(fresnel)
imh0_complex = np.fft.ifft2(pupil_shifted) * (nap**2)
focus_centered = np.fft.fftshift(imh0_complex)
imh0 = np.abs(focus_centered) ** 2

if logger.isEnabledFor(logging.DEBUG):
    plt.figure(figsize=(6, 6))
    plt.imshow(np.abs(fresnel), cmap='gray', origin='lower')
    plt.title(f'Debug at fresnel')
    plt.colorbar()
    plt.xlabel('X [pixels]')
    plt.ylabel('Y [pixels]')
    plt.show()
    plt.clf

    plt.figure(figsize=(6, 6))
    plt.imshow(np.abs(pupil_shifted), cmap='gray', origin='lower')
    plt.title(f'Debug at pupil_shifted')
    plt.colorbar()
    plt.xlabel('X [pixels]')
    plt.ylabel('Y [pixels]')
    plt.show()
    plt.clf

    plt.figure(figsize=(6, 6))
    plt.imshow(np.abs(imh0_complex), cmap='gray', origin='lower')
    plt.title(f'Debug at imh0_complex')
    plt.colorbar()
    plt.xlabel('X [pixels]')
    plt.ylabel('Y [pixels]')
    plt.show()
    plt.clf

    plt.figure(figsize=(6, 6))
    plt.imshow(np.abs(focus_centered), cmap='gray', origin='lower')
    plt.title(f'Debug at focus_centered')
    plt.colorbar()
    plt.xlabel('X [pixels]')
    plt.ylabel('Y [pixels]')
    plt.show()
    plt.clf
    
    plt.figure(figsize=(6, 6))
    plt.imshow(np.abs(imh0), cmap='gray', origin='lower')
    plt.title(f'Debug at imh0')
    plt.colorbar()
    plt.xlabel('X [pixels]')
    plt.ylabel('Y [pixels]')
    plt.show()
    plt.clf

# Intensity normalization
normconst = np.sum(imh0)

# Calculate expected radius using clean centered coordinates
rring = np.sum(imh0 * r) / normconst                             
rradpix2 = np.sum(imh0*r) / np.sum(imh0) # true ring radius in fine pixels
rad = rradpix2/npixperpix  # radius in CCD pixels

print(f'True ring radius [pix]: {rring}')
print(f'True ring radius [arcsec]: {(rring * asperpix)}')
logger.debug(f'rring / ksamp {rring / ksamp}')

# Cube for loop
cube = np.zeros((niter, nccd, nccd), dtype=np.float64)
ix = 0
iy_float = 0

# Progress display step
ndispl = max(niter // 20, 1)
print(f'Computing {niter} iterations...')

# Gif creator
writer = mpl.animation.PillowWriter(fps=5)
tmp_fig = plt.figure()
writer.setup(tmp_fig, 'ringsim.gif', dpi=100)
plt.close(tmp_fig)

# Main Loop
for i in range(niter):
    ix += jstep
    iy_float += jstep * alpha
    iy = math.floor(iy_float)
    
    # Roll if we surpass the center of the array
    if ix > ngrid:
        u1 = np.roll(u1, -ngrid, axis=1)
        ix -= ngrid
    if iy > ngrid:
        u1 = np.roll(u1, -ngrid, axis=0)
        iy_float -= ngrid
        iy = math.floor(iy_float)
    
    uampl = u1[iy : iy+nscr, ix : ix+nscr]
    
    if ksamp > 1:
        zoom_frac = nap / nscr
        uampl_re = zoom(uampl.real, zoom_frac, order=1)
        uampl_im = zoom(uampl.imag, zoom_frac, order=1)
        uampl = uampl_re + 1j * uampl_im
    
    fresnel_uampl = fresnel * uampl
    
    pupil_shifted = np.fft.ifftshift(fresnel_uampl)
    imh1_complex = np.fft.ifft2(pupil_shifted) * (nap**2)
    focus_centered = np.fft.fftshift(imh0_complex if 'imh0_complex' in locals() else imh1_complex)
    imh1 = np.abs(np.fft.fftshift(imh1_complex))**2
    
    # Block-sum (rebin) from fine grid (nap x nap) down to CCD resolution (nccd x nccd)
    impix = imh1.reshape(nccd, npixperpix, nccd, npixperpix).sum(axis=(1, 3)) / normconst
    
    # Optional Jitter / Image Shifts
    if jitter > 0:
        xc = jitter * np.cos(i * omega)
        yc = jitter * np.sin(i * omega)
        # Note: ndshift takes (y_shift, x_shift)
        impix = ndshift(impix, shift=(yc, xc), order=1, mode='nearest')
    
    # --- Photon and Readout Noise (Inside Loop) ---
    if 'starmag' in locals() and starmag != 0.0:
        impix *= starph
        # Ensure non-negative input for Poisson noise generator
        impix_clean = np.maximum(0.0, impix)
        impix = rng.poisson(impix_clean).astype(np.float64) + rng.normal(loc=0.0, scale=ron, size=(nccd, nccd))
    
    # Pupil image for diagnostics
    imh2 = np.abs(uampl * apert)**2
    
    # Save frame to 3D image cube
    cube[i, :, :] = impix
    
    if i % ndispl == 0:
        print(f'Frame {i}/{niter}')
        
        #clear_output(wait=True)  # Clears previous frame before rendering the new one
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))
        
        # Left: Detector Plane
        im1 = ax1.imshow(impix, cmap='gray', origin='lower')
        ax1.set_title(f'Detector Plane (impix) - Frame {i}')
        ax1.set_xlabel('X [pixels]')
        ax1.set_ylabel('Y [pixels]')
        #fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
        
        # Right: Pupil Plane
        im2 = ax2.imshow(imh2, cmap='gray', origin='lower')
        ax2.set_title('Pupil Plane (imh2)')
        ax2.set_xlabel('X [pixels]')
        ax2.set_ylabel('Y [pixels]')
        #fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
        
        plt.tight_layout()
        
        # Gif input
        writer.fig = fig  # Attach current figure instance to writer
        writer.grab_frame()  # Capture frame into GIF buffer
        plt.show()
        plt.clf

print('Simulation done!')
writer.finish()  # Compile output.gif
print('Gif saved')

imav = np.mean(cube, axis=0)

if logger.isEnabledFor(logging.DEBUG):
    plt.figure(figsize=(6, 6))
    plt.imshow(imav, cmap='gray', origin='lower')
    plt.title(f'Debug image average')
    #plt.colorbar()
    plt.xlabel('X [pixels]')
    plt.ylabel('Y [pixels]')
    plt.show()
    plt.clf

    print(f'nap {nap} \n nccd {nccd} \n npixperpix {npixperpix} \n radpix {radpix}')

hdu = fits.PrimaryHDU(cube.astype('float64'))

header = hdu.header
header['DATE-OBS'] = (datetime.datetime.now().strftime('%Y-%m-%dT%H:%M:%S'), 'File creation date / Observation Date')
header['TELESCOP'] = (float(d), 'Telescope Diameter in meters')
header['WAVELEN'] = (float(wavelen), 'Wavelength in meters')
header['PIXSCALE'] = (float(pixel), 'Arcseconds per pixel')
header['EXPOSURE'] = (float(texp*1e6), 'Exposure time in seconds')
header['MAG'] = (float(starmag), 'Stellar magnitude')
header['SEEING'] = (float(seeing), 'Input seeing in arcseconds')
header['CONJ_H'] = (float(pdist), 'Conjugation height in meters')
header['INST'] = (f'{nccd} Simulation', 'Camera dimensions')
# Keys expected by cube2.py
# Note: cube2.py reads EXPOSURE as microseconds (multiplies by 1e-6)
header['GAIN'] = (float(gain), 'Detector gain (cube2.py reads as float)')
header['STAR'] = ('', 'Star name (optional in cube2.py)')

filename = f'{input('Filename to save') or 'test'}.fits'
hdu.writeto(filename, overwrite=True)

print(f'Successfully saved {niter} frames to {filename}')

### cubecoef.pro

In [ ]:
# Process cube of ring-like images

with fits.open("test.fits") as hdul:
    cube = hdul[0].data
    hdr = hdul[0].header

# Out from parameters
m = mmax 
nsect = nsect
interpol = interpol
drad = drad

# Hard-coded
nstart = 50
leak = 1
flat = True

nz, nx, ny = cube.shape
logger.debug(f'sizes {nx, ny, nz}')
if nx != ny:
    raise ValueError(f'Non square frames! ({nx} =/= {ny})')

if nz < nstart:
    raise ValueError(f'Fewer frames than minimum ({nz} < {nstart})')

# Vector radius and coordinates
x_coords = np.arange(nx) - nx / 2
y_coords = np.arange(ny) - ny / 2
x, y = np.meshgrid(x_coords, y_coords)
r = np.hypot(x, y)
phi = np.arctan2(y, x)
phi[ny // 2, : nx // 2 + 1] = -np.pi
x1d = np.arange(nx) # 1D coordinate for interpolation

# Filters and output arrays
rwt = np.zeros((nx, nx, nsect), dtype=np.float64) # Radius calc, check if double nx or nx, ny
fwt = np.zeros((nx, nx, nsect), dtype=np.float64) # Flux in sectors
sect = 2 * np.pi / nsect

# Un masked sectors
for j in range(nsect):
    low_bound = sect * (j - nsect / 2)
    high_bound = sect * (j + 1 - nsect / 2)
    sector = (phi >= low_bound) & (phi <= high_bound) # Bool mask
    fwt[:, :, j] = sector.astype(np.float64)
    rwt[:, :, j] = sector * r

phisect = sect * (np.arange(nsect, dtype=np.float64) - nsect / 2 + 0.5)
xsect = np.cos(phisect)
ysect = np.sin(phisect)

# Initial ring parameters
imav = np.mean(cube[:nstart, :, :], axis=0)
background = float(np.median(np.concatenate([imav[:, 0], imav[:, -1]])))
tmp = imav - background

itot = np.sum(tmp)
xc = np.sum(tmp * x) / itot
yc = np.sum(tmp * y) / itot
# Crude integer recenter: xc shifts columns (axis 1), yc shifts rows (axis 0)
imgcent = np.roll(tmp, (-int(np.round(yc)), -int(np.round(xc))), axis=(0, 1))

radii = np.zeros(nsect, dtype=np.float64)
for j in range(nsect):
    radii[j] = np.sum(imgcent * rwt[:, :, j]) / np.sum(imgcent * fwt[:, :, j])

radpix = np.mean(radii) # Avergae radius
dx1 = np.sum(radii * xsect) / nsect * 2.3 # 2.3 is empirical, this might change
dy1 = np.sum(radii * ysect) / nsect * 2.3
xc += dx1
yc += dy1

# Fourier phase shift centering
arg = 2 * np.pi * (dx1 * x + dy1 * y) / nx
shift_kernel = np.fft.ifftshift(np.exp(1j * arg))
imgcent = np.fft.ifft2(np.fft.fft2(imgcent) * shift_kernel).real

# Diffraction-limited minimum ring width
rwidthmin = (wavelen / d / (1 - eps)) * 2 * 206265 / pixel
tmp_thresh = np.maximum(imgcent - 0.1 * np.max(imgcent), 0) # Image threshold
radvar = np.sum(tmp_thresh * (r - radpix)**2) / np.sum(tmp_thresh)
rwidth = np.sqrt(radvar) * 2.35

print(f'Ring rad, width, minwidth [pix]: {radpix}, {rwidth}, {rwidthmin}')

if radpix > nx / 4: # Check if cube is empty
    impar = {'backgr': backgr, 'flux': -1.0, 'rad': radpix}
    raise ValueError(f'Empty cube')
    #return impar

# Recompute background outside 1.5 ring radius
out = r > 1.5 * radpix
background += float(np.median(imgcent[out]))

# General matrix of masks
drhopix = drad * rwidth
ringmask = (r >= (radpix - drhopix)) & (r <= (radpix + drhopix))
nring = np.count_nonzero(ringmask)

print(f'nring {nring}')

# Full-frame projection matrix: the piston-subtracted cos/sin masks are nonzero
# outside the ring, so projecting against the full (flattened) frame keeps the
# DC-cancellation. Radial/flux masks are zero outside the ring anyway.
ncoef = 2 * nsect + 2 * (m + 1)
maskmat = np.zeros((ncoef, nx * ny), dtype=np.float64)

# Insert radial sector masks into projection matrix
for j in range(nsect):
    maskmat[j, :] = (rwt[:, :, j] * ringmask).ravel()
    maskmat[j + nsect, :] = (fwt[:, :, j] * ringmask).ravel()

cwt = np.zeros((ny, nx, m + 1), dtype=np.float64)
swt = np.zeros((ny, nx, m + 1), dtype=np.float64)

for j in range(m + 1):
    tmp_c = np.cos(phi * j) * ringmask
    cwt[:, :, j] = tmp_c - (np.sum(tmp_c) / (nx * ny) if j > 0 else 0)
    tmp_s = np.sin(phi * j) * ringmask
    swt[:, :, j] = tmp_s - np.sum(tmp_s) / (nx * ny)

    maskmat[2 * nsect + j, :] = cwt[:, :, j].ravel()
    maskmat[2 * nsect + m + 1 + j, :] = swt[:, :, j].ravel()

# Optional control plot before main loop
if display:
    imax = np.max(imgcent[:, nx // 2])
    plt.figure('Sector Definition Control')
    plt.plot(x[0, :], np.maximum(imgcent[:, nx // 2], 0), 'b-', label='Profile')
    plt.plot(-x[0, :], imgcent[:, nx // 2], 'r--', label='Mirrored')
    
    plt.hlines(imax / 2, -radpix - rwidth, -radpix + rwidth, colors='g', linewidth=2, label='Width')
    plt.hlines(imax / 2, radpix - rwidth, radpix + rwidth, colors='g', linewidth=2)
    plt.legend()
    
    plt.title('Initial Ring Alignment Control')
    plt.show()
    plt.clf

# Main Loop over the Cube
coef = np.zeros((ncoef, nz), dtype=np.float64)
xcent = np.zeros(nz, dtype=np.float64)
ycent = np.zeros(nz, dtype=np.float64)
rad = np.zeros(nz, dtype=np.float64)

x0, y0, rad0 = xc, yc, radpix
print('Processing the cube')
imav_sum = np.zeros((ny, nx), dtype=np.float64)

# Gif creator
writer = mpl.animation.PillowWriter(fps=60)
tmp_fig = plt.figure()
writer.setup(tmp_fig, 'cube_centroid.gif', dpi=100)
plt.close(tmp_fig)

for i in range(nz):
    
    tmp_frame = cube[i, :, :].astype(np.float64) - background
    if flat is not None:
        tmp_frame /= flat

    # Frame alignment / sub-pixel shift
    if interpol:
        arg = 2 * np.pi * (x0 * x + y0 * y) / nx
        shift_kernel = np.fft.ifftshift(np.exp(1j * arg))
        tmp_shifted = np.fft.ifft2(np.fft.fft2(tmp_frame) * shift_kernel).real
    else:
        tmp_shifted = np.roll(tmp_frame, (-int(np.round(y0)), -int(np.round(x0))), axis=(0, 1))

    imav_sum += tmp_shifted

    # Project frame onto spatial mask
    c = maskmat @ tmp_shifted.ravel()
    c[:nsect] = c[:nsect] / c[nsect : 2 * nsect]  # Normalize radii
    coef[:, i] = c

    # Update radius and centroids
    radii_i = c[:nsect]
    dr = np.sum(radii_i) / nsect
    dx = np.sum(radii_i * xsect) / nsect * 2.3
    dy = np.sum(radii_i * ysect) / nsect * 2.3

    x0 += leak * dx
    y0 += leak * dy
    rad0 = rad0 * (1 - leak) + dr * leak

    xcent[i] = x0
    ycent[i] = y0
    rad[i] = dr
    
    max_val = np.max(tmp_shifted)
    tmp1 = (tmp_shifted / max_val
            if max_val != 0
            else np.zeros_like(tmp_shifted))
    if (i + 1) % 100 == 0:
        # Overlay sector center coordinates and central pixel marker
        x_pts = np.clip(np.round(radii_i * xsect + nx / 2).astype(int), 0, nx - 1)
        y_pts = np.clip(np.round(radii_i * ysect + ny / 2).astype(int), 0, ny - 1)
        
        tmp_vis = tmp1.copy()
        tmp_vis[y_pts, x_pts] = -0.5
        tmp_vis[int(ny // 2), int(nx // 2)] = 1
        
        #clear_output(wait=True)  # Clears previous frame before rendering the new one
        
        fig = plt.figure('Live Frame Monitor', figsize=(5, 5))
        plt.imshow(tmp_vis, interpolation=None, cmap='gray', origin='lower')
        plt.title(f'Frame {i + 1} / {nz}')
        plt.axis('off')
        plt.pause(0.0001)
        
        # Gif input
        writer.fig = fig  # Attach current figure instance to writer
        writer.grab_frame()  # Capture frame into GIF buffer
        plt.show()
        plt.clf

writer.finish()  # Compile output.gif
print('Gif saved')
    
imav_final = imav_sum / nz

# Post-processing
flux = float(np.mean(coef[2 * nsect, :]))  # m=0 mode mean flux in ADU
coef[2 * nsect :, :] /= flux

fluxvar = float(np.std(coef[2 * nsect, :], ddof=1))
rad_mean = float(np.mean(rad))
xc_mean = float(np.mean(xcent))
yc_mean = float(np.mean(ycent))
xcvar = float(np.std(xcent, ddof=1))
ycvar = float(np.std(ycent, ddof=1))

# Coma
cm = float(np.mean(coef[2 * nsect + 1, :]))
sm = float(np.mean(coef[2 * nsect + mmax + 2, :]))
coma = float(np.hypot(cm, sm))
angle = float(np.degrees(np.arctan2(sm, cm)))

# Ring contrast per sector
contrast = np.zeros(nsect, dtype=np.float64)
for j in range(nsect):
    tmp_sec = imav_final * fwt[:, :, j]
    contrast[j] = np.max(tmp_sec) / np.sum(tmp_sec)

# Noise calc
tmp_norm = imav_final / np.sum(imav_final)
t0 = np.sum(tmp_norm * ringmask)
noise1 = np.sum((ringmask**2) * tmp_norm) / t0  # Angular photon noise factor
noise2 = np.sum(ringmask**2) / t0  # Angular readout noise factor

tmp2 = ringmask * (r - radpix)
noise1r = np.sum(tmp_norm * (tmp2**2))  # Radial photon noise factor
noise2r = np.sum(tmp2**2)  # Radial readout noise factor
noisepar = [float(noise1), float(noise2), float(noise1r), float(noise2r)]

impar = {
    'backgr': background,
    'flux': flux,
    'fluxvar': fluxvar,
    'rad': rad_mean,
    'rwidth': rwidth,
    'xc': xc_mean,
    'yc': yc_mean,
    'xcvar': xcvar,
    'ycvar': ycvar,
    'coma': coma,
    'angle': angle,
    'contrast': float(np.mean(contrast)),
    'noisepar': noisepar}
# This should go in a file to be restored or in a return()

print(f'Cube processed! Parameters saved')


if display:
    plt.figure('Centroid Track (pix)')
    plt.plot(xcent, ycent, '+', label='Centroid (X, Y) [pix]',)
    #plt.plot(ycent * pixel, linestyle='--', label='Y-drift vs Frame')
    plt.axis('equal')
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.title('Centroid Position & Drift [pix]')
    #plt.legend()
    plt.grid(True)
    plt.show()
    plt.clf

### statmom.pro

In [ ]:
# Calculation of statistical moments (matches cube2.py Moments() tail)
m = mmax
nsect = nsect

ncoef, nz = coef.shape
expected_ncoef = 2 * nsect + 2 * m + 2

if ncoef != expected_ncoef:
    raise ValueError(f'Parameters do not match (ncoef={expected_ncoef}, got {ncoef})')

# Differential radius variance across opposite sector pairs (DIMM-like signals)
half_sect = nsect // 2
dr = coef[:half_sect, :] + coef[half_sect:nsect, :]  # (half_sect, nz)
drvar = np.var(dr, axis=1)                           # variances in pix^2
meanrvar = float(np.mean(drvar))

# Radius noise from difference of successive dr values
ddr = dr - np.roll(dr, 1, axis=1)
drnoise = np.var(ddr, axis=1)                        # noise variances, pix^2

# Variance and covariance of angular coefficients
acoef = coef[2 * nsect :, :]                         # (2m+2, nz)
varcoef = np.var(acoef, axis=1)
meancoef = np.mean(acoef, axis=1)
tmp = (acoef * np.roll(acoef, 1, axis=1))[:, 1:]     # shift-1 product, drop first
covar = np.sum(tmp, axis=1) / (nz - 1) - meancoef ** 2

# Add cosine (0..m) and sine (m+1..2m+1) variances and covariances
power = varcoef[: m + 1] + varcoef[m + 1 : 2 * m + 2]
cov = covar[: m + 1] + covar[m + 1 : 2 * m + 2]

# Mean coefficients for aberrations (mean radii and m=1,2,3 cos/sine terms)
mcoef = np.zeros(nsect + 6, dtype=np.float64)
mcoef[:nsect] = np.mean(coef[:nsect, :], axis=1)
mcoef[nsect : nsect + 3] = meancoef[1:4]
mcoef[nsect + 3 : nsect + 6] = meancoef[m + 2 : m + 5]

if display:
    arg = np.arange(m + 1)

    plt.figure('Angular Power Spectrum', figsize=(7, 5))
    plt.semilogy(arg, power, 'k-o', label='Power Spectrum')
    plt.semilogy(arg, cov, 'r--', label='Covariance')
    plt.xlabel('m')
    plt.ylabel('Power')
    plt.title('Angular Mode Spectrum')
    plt.legend()
    plt.grid(True)
    plt.show()
    plt.clf

# Assemble output dictionaries (same structure as cube2.py)
moments = {
    'var': power.tolist(),
    'cov': cov.tolist(),
    'rnoise': float(drnoise[0]),
    'rvar': meanrvar,
    'mcoef': mcoef.tolist()}

data = {'image': {'impar': impar, 'noisepar': noisepar}, 'moments': moments}

# Assemble <par> from the sim1.par-cell variables for next cells
par = {
    'telescope': {
        'D': d, 'eps': eps, 'pixel': pixscale, 
        'ringradpix': ringradpix, 'ron': ron},
    'profrest': {
        'mmax': mmax,
        'wav': [600],
        'sp':  [1.0],   # flat single-wavelength response -- matches monochromatic wav=[600] (sim wavelen=0.6e-6 m)
        'weightfile': 'weights.json',
    },
}
print('Par file created with',
        'D:', d, 'eps:', eps, 'pixel:', pixscale, 
        'ringradpix:', ringradpix, 'ron:', ron)

### aweight.pro
python

In [ ]:
def aweight(z,mmax,d,eps,pdist,wav,sp,drho=1.5,pixel=0,zn=[],zrad=[]):
    # Computing paramters hard coded
    nsect = 8     # number of sectors for radial weight
    ngrid = 512   # half-size of computing grid [pix]
    ksize = 6     # grid size/telescope diameter ratio

    nz = z.shape[0]
    nwav = wav.shape[0]  # nwav is number of wavelengths
    spnorm = sp/np.sum(sp) # normalize the spectrum
    lam0 = np.sum(wav*spnorm) # average wavelength
    if nwav > 1:
        wavstep = wav[1] - wav[0] # step of wavelength grid, assumed uniform
    else:
        wavstep = 0

    # Define main calculation parameters
    size = d * ksize  # domain size in pupil plane, [m]
    asperpix = 206265 * lam0 / size  # fine-pixel in the image plane [arcsec]
    fstep = 1 / size  # frequency step [1/m]
    xstep = size / (2*ngrid)  # pixel in the pupil plane
    ringradpix = 0.85 * d * (1+eps) / (4*pdist) * 206265 / asperpix # ring radius in fine pixels
    
    if ringradpix > 0.8 * ngrid:
        print("Error: ring too wide, returning!")
        

    # Prepare the arrays
    i = np.indices((2 * ngrid, 2 * ngrid))
    x = i[1] - ngrid
    y = i[0] - ngrid
    r = np.sqrt( np.square(x) + np.square(y))
    r[ngrid,ngrid] = 1e-3 # to avoid division by zero
    phi = np.arctan2(y,x)  # 2D array of phase # plt.imshow(phi, cmap='Greys')

    # Define the annular aperture in the pupil space
    radpix = ngrid*d/size  # aperture radius, pixels
    pupil = (r <= radpix) * (r >= eps*radpix) # boolean array, true inside pupil
    ninside = np.sum(pupil)  # pupil surface [pix]

    # Define conic wavefront at the pupil
    a4 = d**2 / (lam0 * pdist) * (np.pi / 8 * 3**(-0.5))   # Zernike defocus corresponding to the propagation distance [rad]
    a11 = -0.1 * a4       # spherical aberration coef. [rad]
    tmp = a11 * 5**(0.5) * (6 * (r/radpix)**4  - 6 * (r/radpix)**2)
    tmp = tmp + a4 * 2 * 3**0.5 * ((r/radpix)**2 - 0.5)  # wavefront shape [rad]
    
    # Optionally add Zernike aberrations
    nzern = len(zn)
    for j in range(0,nzern):
        tmp += zrad[j]*zernike.zernikel(zn[j],r/radpix,phi)


    uampl = pupil*(np.cos(tmp) + np.sin(tmp)*1j) # complex amplitude at the pupil

    # Compute nominal ring image at the focal plane
    imh = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(uampl)))
    imh = np.power(np.abs(imh),2)/( (2 * ngrid)**2) # np.sum(imh) = ninside to check normalization

    # Compute the masks
    ringradpix2 = np.sum(imh * r) / ninside # true ring radius in fine pixels
    ringrad = ringradpix2 * asperpix / pixel  # ring radius in CCD pixels
    print ("Ring radius [pix]: ",ringrad)
    drhopix = drho * lam0 / d / (1-eps) * 2 * 206265 / asperpix # ring half-width [pix]
    filtap = (r >= ringradpix2 - drhopix) * (r <= ringradpix2 + drhopix) # radial part of image mask
    nm = mmax + 1  # number of angular coefficients
    wt = np.zeros((nz,nm))
    ufunc = np.zeros((nz,nm))

    # Prepare things used in the loop
    spturb = np.power(r, -11 / 3 ) * fstep**(-5 / 3) * ( 0.5 * 9.62 / np.pi ) * lam0**(-2) # turbulence phase spectrum for Jturb=1
    spturb[ngrid,ngrid] = 0
    spufunc = spturb * np.square(np.pi * fstep * r)  # for U-function calculation
    flux = np.sum(imh * filtap) # flux inside the ring mask
    utmp = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(np.conj(uampl)))) # auxiliary conjugated amplitude
    #utmp = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(np.conj(uampl.copy())))) # auxiliary conjugated amplitude

    # Radial mask for differential sector motion
    sectrad = np.pi / nsect # sector width [rad] = 22.5deg for nsect=8
    tmp = np.mod(phi + np.pi, np.pi)  # 180-folded phase
    sector = (tmp >= np.pi / 2 - sectrad) * (tmp < np.pi / 2 + sectrad) # 1 within opposite 45-deg sectors
    sector = np.transpose(sector) # rotate 90 degrees to match IDL, sectors along X
    ringmask = filtap * sector
    rflux = np.sum(imh * ringmask) # 1/4 of full flux
    rwt = r * ringmask
    tmp = np.sum(imh * rwt) / rflux  # mean radius, to make np.sum(imh*rwt)=0
    rwt = rwt - tmp * ringmask # subtract to get zero signal without turbulence
    rwt = rwt / ksize  # radius in lam/D units instead of fine pixels

    # Prepare 2D Fresnel filters for propagation calculation
    frecos = np.zeros((nz, 2 * ngrid, 2 * ngrid))
    fresin = np.zeros((nz, 2 * ngrid, 2 * ngrid))
    for iz in range(0,nz):
        zdist = z[iz]
        damp1 = np.exp(-np.square(0.5 * np.square(r * fstep) * wavstep * zdist)) #0.5 damping factor
        spcos = np.zeros((2 * ngrid, 2 * ngrid))
        spsin = np.zeros((2 * ngrid, 2 * ngrid))
        for j in range(0,nwav):
            w = wav[j]
            arg = np.pi * w * zdist * np.square(r * fstep)
            a = spnorm[j] * lam0 / w
            spcos += a * np.cos(arg)
            spsin += a * np.sin(arg)
        frecos[iz,:,:] = spcos / spcos[ngrid,ngrid] * damp1
        fresin[iz,:,:] = spsin / spcos[ngrid,ngrid] * damp1

    # Compute the weights, loops in m and z
    for m in range(0,nm):  # loop over m
        if m==0: # radial weight
            cwt = rwt
            swt = 0
            pixfact = 1.
        else:
            cwt = np.cos(phi * m) * filtap # cosine mask
            swt = np.sin(phi * m) * filtap # sine mask
            if pixel > 0:   # pixel averaging factor
                arg = pixel / asperpix / (1.5 * ringradpix2) * m
                pixfact = np.sin(arg) / arg
            else:
                pixfact = 1
        # Normalize by sector or total flux
        if m==0:
            normfact = 1 / rflux # for radial weight
        else:
            normfact = 1 / flux  # for angular weight
        # Response to the cosine mask
        result1 = uampl * np.fft.fftshift(np.fft.ifft2(np.fft.fftshift(utmp * cwt)))
        result1 *= np.square(2 * ngrid)
        fphase1 = -result1.imag * normfact
        fampl1 = result1.real * normfact
        # Response to the sine mask
        if m>0:
            result2 = uampl * np.fft.fftshift(np.fft.ifft2(np.fft.fftshift(utmp * swt)))
            result2 *= np.square(2 * ngrid)
            fphase2 = -result2.imag * normfact
            fampl2 = result2.real * normfact
        else:
            fphase2 = fampl2 = 0
        # Propagation filter
        for iz in range(0,nz):   # loop over distance grid
            zdist = z[iz]
            ctmp = frecos[iz,:,:]
            stmp = fresin[iz,:,:]
            # Propagate cos/sin filters over distance z
            tmp1 = np.fft.fftshift(np.fft.ifft2(np.fft.fftshift(fphase1))) * ctmp
            tmp1 += np.fft.fftshift(np.fft.ifft2(np.fft.fftshift(fampl1))) * stmp
            if m>0:
                tmp2 = np.fft.fftshift(np.fft.ifft2(np.fft.fftshift(fphase2))) * ctmp
                tmp2 +=  np.fft.fftshift(np.fft.ifft2(np.fft.fftshift(fampl2))) * stmp
            else:
                tmp2 = 0
            pfilter = np.power(np.abs(tmp1),2) + np.power(np.abs(tmp2),2)
            wt[iz,m] = np.sum(pfilter * spturb) * pixfact
            ufunc[iz,m] = np.sum(pfilter * spufunc)
###  End of the weight-calculation loop over m and z
    return wt, ufunc, ringrad

### getweight5.pro

In [ ]:
# Return blackbody spectrum with arbitrary normalization, photons/lambda
# wavelength in m, temperature in K
def blackbody(wav,temp):
    const = 0.014387618 # in [m.K]
    planck = np.power(wav[0] / wav,4) / (np.exp(const / wav / temp) -1)
    return planck / np.max(planck)
# Find U-coefficients
def getucoef(ufunc,z,mm):
    nm = len(mm)
    nz = z.shape[0] -2 # number of layers, exclude 2 lowest
    amat = np.zeros((nm,nz)) # Matrix of the inear-equations system
    # wz = z[2:nz+2]*1e-3 + 0.5 # distance-dependent weight for response calc.
    wz = np.zeros(nz) + 1. # distance-independent weight
    for i in range(0,nm):
        amat[i,:] = ufunc[2:nz+2,mm[i]]*wz
# Least-squares system
    aa = np.dot(amat,np.transpose(amat)) # 6x6 square matrix
    bb = np.dot(amat,wz) # 6-element vector of right-hand terms
    ainv = np.linalg.pinv(aa,1e-4)  # SVD inversion with 1E-4 threshold
    ucoef = np.dot(ainv,bb)
    uresp = np.zeros(nz) # resulting response, must be close to one at all z>0
    for i in range(0,nm):
        uresp = uresp + ucoef[i]*ufunc[2:nz+2,mm[i]]
    #plt.plot(z,uresp)
    return ucoef, uresp

# read parameters, return the dictionary <par>
def getpar(parfile):
    try:
        file = open(parfile, "r")
        par = json.load(file)   # par is a nested dictionary
    except FileNotFoundError as err:
        print(err)
        quit()
    file.close()
    return par

def computeweight(par):  # actual weight calculation
    d = float(par["telescope"]["D"])
    eps = float(par["telescope"]["eps"])
    #pdist = par["telescope"]["pdist"]
    pixel = float(par["telescope"]["pixel"])
    ringradpix = float(par["telescope"]["ringradpix"])
    wav = np.array(par["profrest"]["wav"])*1e-9 # wavelength in m
    sp = np.array(par["profrest"]["sp"]) # spectral response
    sp0 = sp * blackbody(wav, 10213) # B-V=0 spectrum
    sp1 = sp * blackbody(wav, 03938) # B-V=1 spectrum
    sp0 = sp0 / np.sum(sp0) # normalize
    sp1 = sp1 / np.sum(sp1)
    lameff = (np.sum(wav * sp0), np.sum(wav * sp1)) # effective wavelength
    mmax = par["profrest"]["mmax"]
    if "aber" in par["profrest"]:
        adict = getpar(par["profrest"]["aber"])
        zn = [2,3,4,5,6,7,8,9,10]
        zrad = np.zeros(9)
        zrad[2:9] = np.array(adict["zampl"], float) # Focus to trefoil
        zrad[0] =  -5 * zrad[6]
        zrad[1] = -5 * zrad[5]
        s = "Zrad:  "
        for i in range(0,7):
            s += " {:.3f}".format(zrad[i])
        #print(s)
        #print(zrad)
    else:
         zn = zrad = []


# distance grid, log-spaced with sqrt(2) step, 0.25-32km
    nz = 16
    z = np.zeros(nz)
    z[1:nz] = 1e3 * 2**(0.5 * np.arange(nz-1) -2)

# Find propagation distance from ring radius. Use analytic approx. first
    HR = 0.85 * d * (1 + eps) / 4 / pixel * 206265
    #HR = 0.85 * d * (1 + eps) / (4 * pdist) * 206265 / asperpix
    print("Initial H*R [m.pix]: {:.2f} ".format(HR))
    pdist = HR / ringradpix
    wt0, ufunc0, ringrad = aweight(np.zeros(1),1,d,eps,pdist,wav,sp0,1.5,pixel,zn,zrad)
    HR1 = ringrad * pdist
    pdist = HR1 / ringradpix
    print("Final H*R [m.pix] and pdist: {:.2f} {:.2f} ".format(HR1,pdist))

    # return

# Weight for B-V=0
    print("Computing weight for B-V=0...")
    wt0, ufunc0, ringrad = aweight(z,mmax,d,eps,pdist,wav,sp0,1.5,pixel,zn,zrad) # arrays of [nz,mmax+1] dimension
    hslope = pdist*ringrad
    print("Ring radius and H*rad [m.pix]: {:.3f} {:.3f} ".format(ringrad, hslope))
# Weight for B-V=1
    print("Computing weight for B-V=1...")
    wt1, ufunc1, ringrad1 = aweight(z,mmax,d,eps,pdist,wav,sp1,1.5,pixel,zn,zrad) # arrays of [nz,mmax+1] dimension

    mm = [1,3,6,7,8,9] # selected frequencies for wind measurement
    ucoef0, resp0 = getucoef(ufunc0,z,mm)
    ucoef1, resp1 = getucoef(ufunc1,z,mm)
    #plt.plot(resp0)
    #plt.plot(resp1)
# Color dependence
    wtslope = wt1 - wt0
    ucoefslope = ucoef1 - ucoef0

# serialize and save the weights
    weight = {
        "z":z.tolist(),"wt0":wt0.tolist(),
        "wtslope":wtslope.tolist(),"ucoef0":ucoef0.tolist(),
        "ucoefslope":ucoefslope.tolist(),"umm":mm,"lameff":lameff,
        "ringrad":ringrad,"pdist":pdist
    }

    json_output_file = par["profrest"]["weightfile"]
    try:
        json.dump(weight, codecs.open(json_output_file, 'w', encoding='utf-8'), separators=(',', ':'), sort_keys=True, indent=4)
    except FileNotFoundError as err:
        print('{}:{}'.format(err, json_output_file))
    #print("Saved weights in "+json_output_file)
    return weight

weight = computeweight(par)
print("Weights computed and saved to weights.json")
print("  z-layers = {},  coeffs/layer = {}".format(len(weight["z"]), len(weight["wt0"][0])))
print("  ring radius [pix] = {:.3f},  pdist [m] = {:.2f}".format(weight["ringrad"], weight["pdist"]))

### testsimul

In [ ]:
# Noise on the angular and radial coefficients
flux = impar['flux']                                   # star flux, ADU per frame
anoise = noisepar[0] / flux + noisepar[1] * (ron / flux) ** 2  # noise variance of a-coef
flux1 = flux / nsect                                   # flux per sector
rnoise1 = 2 * (noisepar[2] / flux1 + noisepar[3] * (ron / flux1) ** 2 / nsect)  # radius noise, pix^2

# IDL-named weight arrays pulled out of the getweight5 'weight' dict
z = np.array(weight['z'])              # nominal weight-distance grid [m] (len 16)
wt0 = np.array(weight['wt0']).T        # transpose to (m, nz) to match IDL wt0[m, z]
ucoef0 = np.array(weight['ucoef0'])
mm = weight['umm']

# Angular power spectrum and covariance from statmom
powspec = np.array(moments['var'], float)
covspec = np.array(moments['cov'], float)

# Standard altitude layers (0, 0.25 km, 0.5 km, 1 km ... 16 km)
nz = 8
z0 = np.concatenate(([0], 1000 * (2 ** (np.arange(nz - 1) - 2))))

# Noise-subtract the power and get the scintillation index (testsimul.pro lines 66-67)
powspec = np.maximum(powspec - anoise, 0)
totvar = float(np.sum(powspec))

# Interpolate the weighting functions onto the z0 altitude grid
m = wt0.shape[0]                       # number of m-terms (mmax+1)
wt = np.zeros((m, nz), dtype=np.float64)
for i in range(m):
    wt[i, :] = np.interp(z0, z, wt0[i, :])

# Radius rms from statmom (used as radvar = rrms**2 - rnoise1 in the report cell)
rrms = np.sqrt(meanrvar)

# Plot angular power spectrum, covariance and noise floor
if display:
    arg = np.arange(len(powspec))
    plt.figure('Angular power / covariance / noise', figsize=(7, 5))
    plt.semilogy(arg, np.maximum(np.array(moments['var'], float), 1e-12), 'k-o', label='power')
    plt.semilogy(arg, np.maximum(covspec, 1e-12), 'r--', label='covariance')
    plt.axhline(max(anoise, 1e-12), ls=':', color='b', label='noise')
    plt.xlabel('m'); plt.ylabel('Power'); plt.title('Angular power spectrum')
    plt.legend(); plt.grid(True); plt.show()

### profrest5.pro
also in python

In [ ]:
# read a json file, return a dictionary; returns None if fails
def read_json(filename):
    try:
        file = open(filename, "r")
        p = json.load(file)
        file.close()
        return p
    except FileNotFoundError as err:
        print(err)
        return


# Profile restoration. Inputs: dictionaries of parameters, data, weights, and Z-matrix
# Output: dictionary of profile parameters
# Before calling Restore, run getzen.py to define the zenith distance and star color in data
#
def Restore(par, data, weight, zmat):
    #    print(data["image"]["impar"])
    var = data["moments"]["var"]  # variance of a-coefficients
    cov = data["moments"]["cov"]  # covariance of a-coefficients
    impar = data["image"]["impar"]  # ring parameters
    noisepar = data["image"]["noisepar"]  # noise parameters
    starpar = data["starpar"]  # star parameters
    zen = starpar["zen"]
    bv = starpar["BV"]
    z0 = par["profrest"]["zgrid"]  # grid of heights representing the turbulence profile

    z = np.array(weight["z"])  # nominal distance grid of weights
    nz = len(weight["z"])  # number of layers in weights
    nm = len(weight["wt0"][0])  # number of coefficints per layer
    wt = np.reshape(np.array(weight["wt0"]), (nz, nm))  # wt.shape = (16,21)
    wt += bv * np.reshape(np.array(weight["wtslope"]), (nz, nm))  # must be >0!

    # interpolate weight to the Z0 grid, trim to mmax
    cosz = np.cos(zen / 180 * np.pi)  # cos(z)
    mmax = 15  # hard-coded number of terms to use
    nz0 = len(z0)
    z00 = np.array(z0) / cosz  # stretched array of nominal heights
    wt1 = np.zeros((nz0, mmax))  # interpolated and trimmed weights
    for m in range(0, mmax):
        wt1[:, m] = np.interp(z00, z, wt[:, 1 + m])
    wtsect = np.interp(z00, z, wt[:, 0])  # sector-motion weight on z0 grid

    # Check that moments and weights have the same number of coefficients
    mcoef = len(var)  # 21 for mmax=20
    if mcoef != nm:
        print("Error! Mismatching number of coefficients: ", mcoef, nm)
        return

    # noise bias, see allcubes5.pro => noisecubes
    gain = data["cubepar"]["gain"]  # electrons per ADU
    #eladu = 3.60 * pow(10, -gain / 200)
    eladu = 1.0 # sim counts photo-electrons directly (gain=0 in sim1.par => ADU == electrons); was 0.3 for a real gain=200 camera
    noisepar = data["image"]["noisepar"]  # list of 4 numbers
    fluxadu = float(data["image"]["impar"]["flux"])
    #    print(fluxadu)
    flux = eladu * fluxadu  # flux in electrons

    anoise = float(noisepar[0]) / flux + float(noisepar[1]) * pow(par["telescope"]["ron"] / flux,
                                                                  2)  # noise variance of a-coef
    rnoise = 2. * (float(noisepar[2]) / flux + float(noisepar[3]) * pow(par["telescope"]["ron"] / flux,
                                                                        2) / 8)  # noise on radius, pix^2

    #    anoise = noisepar[0]/flux + noisepar[1]*pow(par["telescope"]["ron"]/flux,2) # noise variance of a-coef
    #    rnoise = 2.*(noisepar[2]/flux + noisepar[3]*pow(par["telescope"]["ron"]/flux,2)/8) # noise on radius, pix^2

    # Select max frequency used in the restoration
    var1 = np.array(var[1:mmax + 1], float) - anoise
    var1 *= (var1 > 0)  # prevent negative (sets negative values to zero)
    cov1 = np.array(cov[1:mmax + 1], float)
    rho = cov1 / var1
    varcorr = var1 / (0.8 + 0.2 * rho)  # correction for finite exposure time
    totvar = np.sum(np.array(var, float))  # full scintillation power

    # Z-matrix correction. Avoid negative values!
    z1 = np.array(zmat)
    ncol = z1.shape[1]
    var2 = np.array(var[1:ncol + 1], float)  # indices used for correction
    varz = varcorr / (1. + np.dot(z1[0:mmax, :], var2))  # correct for saturation

    # weighted nnls. Weight is proportional to 1/var (before noise subtraction, non-negative)
    varwt = np.power(np.array(var[1:mmax + 1], float), -1)
    a2 = np.transpose(wt1.copy())  # matrix of (mmax-1,nz) dimension
    for i in range(0, mmax):
        a2[i, :] *= varwt[i]
    varz2 = varz * varwt
    prof, resvar = optimize.nnls(a2, varz2)  # turbulence intergals in  [m^1/3] in z0 layers

    varmod = np.dot(a2, prof) / varwt
    erms = np.std(1. - varmod / varz)
    #print("RMS residual: {:.3f}".format(erms))

    prof1 = prof * cosz  # zenith-corrected integrals
    jtot = np.sum(prof1)  # turbulence integral
    seeconst = 6.83e-13  # integral for 1" seeing at 500nm
    see = pow(jtot / seeconst, 0.6)  # seeing in arcseconds
    jfree = np.sum(prof1[2:nz0])  # start at 0.5km
    fsee = pow(jfree / seeconst, 0.6)  # FA seeing in arcseconds

    # Compute seeing from radius var.
    d = par["telescope"]["D"]
    pixel = par["telescope"]["pixel"]
    lameff = weight["lameff"]
    lam0 = lameff[0] + bv * (lameff[1] - lameff[0])  # effective wavelength for the star color
    rvar = float(data["moments"]["rvar"]) - rnoise  # radius variance in pix^2, noise-subtracted
    lamd = lam0 / d * 206265.  # lambda/D in arcseconds
    rvarnorm = rvar * pow(pixel / lamd, 2)  # variance in (lam/D^2) units

    wcoef = np.sum(wtsect * prof) / np.sum(prof)  # profile-adjusted weight of sector variance
    jtot2 = rvarnorm / wcoef / 4  # turbulence intergal, m^1/3. Explain factor 4!
    see2 = pow(jtot2 * cosz / seeconst, 0.6)  # seeing at zenith, arcsec
    see2 *= 1. / (1. - 0.4 * totvar)  # saturation correction
    #print("Seeing (sect,tot,FA): {:.3f} {:.3f} {:.3f}".format(see2, see, fsee))

    # Wind measurement
    texp = 1e-3  # hard-coded exposure time
    delta = (var1 - cov1) * (texp ** (-2))
    delta = delta * (delta > 0)
    ucoef = np.array(weight["ucoef0"]) + bv * np.array(weight["ucoefslope"])
    umm = weight["umm"]
    delta = np.array(delta[umm])
    v2mom = np.sum(ucoef * delta)
    jwind = np.sum(prof[1:nz0])  # exclude ground layer, the speed is not zen-corrected
    v2 = pow(v2mom / jwind, 0.5)  # use profile uncorrected for zenith
    #print("Wind speed [m/s]: {:.3f}".format(v2))

    # tau_0 at 500nm
    r0 = pow(6.680e13 * jwind, -0.6)
    tau0 = 310. * r0 / v2  # in [ms]
    # isoplanatic angle at 500nm at zenith
    r0 = 0.101 / see
    tmp = pow(np.array(z0), 1.6667)
    heff = pow(np.sum(tmp * prof1) / np.sum(prof1), 0.6)
    theta0 = 206265 * 0.31 * r0 / heff
    # Output dictionary
    profile = {"z0": z0, "prof": (prof * 1e13).tolist(), "see": see, "fsee": fsee, "see2": see2, "wind": v2, "erms": erms,
               "totvar": totvar, "tau0": tau0, "theta0": theta0}

    return profile


# ### Driver: restore the turbulence profile from the notebook's data/weight
# profrest5 needs a Z-matrix (saturation correction) plus two sub-dicts the
# simulation does not produce, so we supply them here:
#   starpar : simulated star observed at zenith (zen=0), color B-V = bv below
#   cubepar : detector gain (eladu is hard-coded inside Restore anyway)
import os

zmat = read_json(os.path.join('andrei', 'zmat.json'))   # (20,5) saturation matrix, La Serena

# Heights [m] for the restored profile (same as testsimul.pro z0 / par-laserena zgrid)
par["profrest"]["zgrid"] = [0, 250, 500, 1000, 2000, 4000, 8000, 16000]

# Observing geometry of the simulated star
bv = 0.0                                    # star color B-V (0 = blue)
data["starpar"] = {"zen": 0.0, "BV": bv}    # simulated at zenith
data["cubepar"] = {"gain": gain}            # detector gain from the sim1.par cell

profile = Restore(par, data, weight, zmat)

# Expose IDL-style names for the "remaining testsimul code" cell.
# Restore returns prof scaled by 1e13; convert back to m^1/3 for the .pro-style math.
prof = np.array(profile["prof"], float) / 1e13
wind = float(profile["wind"])
erms = float(profile["erms"])
print("Profile restored (report printed by the 'remaining testsimul code' cell).")


### remaining testsimul code

In [ ]:
# --- testsimul.pro : after profrest5 (final seeing + report) ---
# Uses prof/wind (from the profrest5 cell) and anoise-corrected totvar, z0, wt,
# rrms, rnoise1 (from the 'testsimul inbetween' cell). prof is in m^1/3.

# 1. Total and free-atmosphere turbulence integrals
jtot = np.sum(prof[:nz])
see = (jtot / 6.826e-13) ** 0.6
jfree = np.sum(prof[2:nz])
fsee = (jfree / 6.8e-13) ** 0.6
print(f"See, fsee:    {see:6.2f}{fsee:6.2f}")
print(f"Scint: {totvar:6.3f}{totvar:10.2e}")

# 2. Formatted array outputs matching IDL's (A12, 10F7.2) specifier
z_str = "".join([f"{v:7.2f}" for v in (z0 * 1e-3)[:10]])
j_str = "".join([f"{v:7.2f}" for v in (prof * 1e13)[:10]])
print(f"{'Z [km]:':<12}{z_str}")
print(f"{'J [1e-13]:':<12}{j_str}")
print(f"Wind: {wind:7.1f}")

# 3. Alternative seeing from sector-radius variance
wtsect = np.sum(wt[0, :] * prof) / np.sum(prof)   # profile-weighted sector weight
lamd = (wavelen / d) * 206265.0                   # lambda/D in arcsec
radvar = rrms ** 2 - rnoise1                       # noise-corrected radius variance, pix^2
rvarnorm = radvar * (pixscale / lamd) ** 2         # variance in (lambda/D)^2 units
jtot2 = rvarnorm / wtsect / 4.0                    # zenith turbulence integral
see2 = (jtot2 / 6.826e-13) ** 0.6                  # seeing in arcsec
see2 = see2 / (1.0 - 0.40 * totvar)                # scintillation saturation correction
print(f"Sector seeing: {see2:6.2f}")

print(f"Input seeing and J: {seeing:6.2f}{tint * 1e13:6.2f}")
print(f"Altitudes: {zlow:8.1f}{zhigh:8.1f}")
print("Simulated cube is processed!")